# VGGT Streaming Variants Comparison

**Goal:** Benchmark 5 VGGT variants on latency, peak GPU memory, and output shape consistency
to select the optimal encoder backbone for the thesis pipeline (world model + 3D scene understanding).

**Variants:**

| # | Variant | Key Feature | Repo |
|---|---------|-------------|------|
| 1 | VGGT (baseline) | Full 1B model, batch forward | `facebook/VGGT-1B` |
| 2 | StreamVGGT | Temporal streaming with KV cache | `lch01/StreamVGGT` |
| 3 | SceneVGGT | SLAM integration, scene modules | Same weights as VGGT |
| 4 | InfiniteVGGT | Token budget for long sequences | StreamVGGT weights |
| 5 | FastVGGT | Token merging for speed | `facebook/VGGT_tracker_fixed` |

**Acceptance criteria:** latency measurements for N=10,20,50,100,500 frames, peak GPU memory,
output shape consistency, summary DataFrame, visualization plots.

**Kernel:** uv-managed Python 3.12 env (`uv run jupyter lab`)

---
## Table of Contents
1. [Setup & Imports](#1)
2. [Collect / Generate Observations](#2)
3. [Model Loading Functions](#3)
4. [Benchmark Function](#4)
5. [Per-Variant Inference Wrappers](#5)
6. [Run Benchmarks](#6)
7. [Results DataFrame](#7)
8. [Latency Plot](#8)
9. [Memory Plot](#9)
10. [Combined Summary Bar Chart](#10)
11. [Output Shape Consistency](#11)
12. [Summary & Recommendation](#12)

---
## 1. Setup & Imports <a id='1'></a>

In [ ]:
import sys
import os
import gc
import time
import importlib
from pathlib import Path

import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

REPO_ROOT = Path(os.path.abspath(".."))
EXTERNAL = REPO_ROOT / "external"

# Device detection
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

# Precision: bfloat16 if compute capability >= 8 (Ampere+), else float16
if device.type == "cuda":
    cc = torch.cuda.get_device_capability()
    dtype = torch.bfloat16 if cc[0] >= 8 else torch.float16
else:
    dtype = torch.float32

print(f"torch: {torch.__version__}")
print(f"device: {device}")
print(f"dtype: {dtype}")
if device.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name()}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

In [ ]:
# GPU memory helpers
def reset_memory_stats():
    """Reset CUDA peak memory tracking."""
    if device.type == "cuda":
        torch.cuda.reset_peak_memory_stats()
        torch.cuda.empty_cache()

def get_peak_memory_mb() -> float:
    """Return peak GPU memory allocated in MB since last reset."""
    if device.type == "cuda":
        return torch.cuda.max_memory_allocated() / 1e6
    return 0.0

def unload_model(model):
    """Delete model and free GPU memory."""
    del model
    gc.collect()
    if device.type == "cuda":
        torch.cuda.empty_cache()

---
## 2. Collect / Generate Observations <a id='2'></a>

Try to load Habitat observations; fall back to synthetic random tensors if `habitat_sim` is unavailable.

In [ ]:
SEQ_LENGTHS = [10, 20, 50, 100, 500]
IMG_H, IMG_W = 518, 518  # VGGT default input size
MAX_N = max(SEQ_LENGTHS)

USE_HABITAT = False
try:
    import habitat_sim
    USE_HABITAT = True
except ImportError:
    pass

if USE_HABITAT:
    print("habitat_sim available — collecting real observations")
    scene_dataset = REPO_ROOT / "data" / "scene_datasets" / "hm3d" / "minival"
    scene_paths = sorted(scene_dataset.rglob("*.glb"))
    if not scene_paths:
        print(f"WARNING: No .glb scenes in {scene_dataset}, falling back to synthetic")
        USE_HABITAT = False

if USE_HABITAT:
    # Collect RGB frames from first HM3D scene with random actions
    sim_cfg = habitat_sim.SimulatorConfiguration()
    sim_cfg.scene_id = str(scene_paths[0])
    sim_cfg.enable_physics = False

    rgb_cfg = habitat_sim.CameraSensorSpec()
    rgb_cfg.uuid = "rgb"
    rgb_cfg.sensor_type = habitat_sim.SensorType.COLOR
    rgb_cfg.resolution = [IMG_H, IMG_W]
    rgb_cfg.position = [0.0, 1.5, 0.0]

    agent_cfg = habitat_sim.agent.AgentConfiguration(sensor_specifications=[rgb_cfg])
    sim = habitat_sim.Simulator(habitat_sim.Configuration(sim_cfg, [agent_cfg]))

    import random
    action_names = ["move_forward", "turn_left", "turn_right", "look_up", "look_down"]
    frames = []
    sim.reset()
    for _ in range(MAX_N):
        obs = sim.get_sensor_observations()
        rgb = obs["rgb"][:, :, :3]  # drop alpha, [H, W, 3] uint8
        # Convert to [3, H, W] float32 [0, 1]
        frame = torch.from_numpy(rgb).permute(2, 0, 1).float() / 255.0
        frames.append(frame)
        sim.step(random.choice(action_names))
    sim.close()

    all_images = torch.stack(frames)  # [MAX_N, 3, H, W]
    print(f"Collected {len(frames)} Habitat frames: {all_images.shape}")
else:
    print("WARNING: habitat_sim not available — using synthetic random tensors")
    print("Results will NOT reflect real-world performance characteristics.")
    rng = np.random.default_rng(42)
    all_images = torch.from_numpy(
        rng.random((MAX_N, 3, IMG_H, IMG_W), dtype=np.float32)
    )
    print(f"Generated synthetic frames: {all_images.shape}")

print(f"Value range: [{all_images.min():.3f}, {all_images.max():.3f}]")
print(f"dtype: {all_images.dtype}")

---
## 3. Model Loading Functions <a id='3'></a>

Each variant has its own loader. Since VGGT/SceneVGGT/FastVGGT share the `vggt` module name,
we swap `sys.path` entries and use `importlib` to avoid import collisions.

In [ ]:
def _clean_sys_path():
    """Remove all external/* entries from sys.path to avoid import collisions."""
    ext_str = str(EXTERNAL)
    sys.path[:] = [p for p in sys.path if not p.startswith(ext_str)]
    # Also remove cached vggt/streamvggt modules
    for mod_name in list(sys.modules.keys()):
        if mod_name.startswith("vggt") or mod_name.startswith("streamvggt"):
            del sys.modules[mod_name]


def load_vggt_baseline():
    """Load VGGT baseline (facebook/VGGT-1B)."""
    _clean_sys_path()
    sys.path.insert(0, str(EXTERNAL / "VGGT"))
    from vggt.models.vggt import VGGT
    model = VGGT.from_pretrained("facebook/VGGT-1B")
    model = model.to(device).eval()
    return model


def load_streamvggt():
    """Load StreamVGGT (lch01/StreamVGGT)."""
    _clean_sys_path()
    sys.path.insert(0, str(EXTERNAL / "StreamVGGT" / "src"))
    from streamvggt.models.streamvggt import StreamVGGT
    from huggingface_hub import hf_hub_download
    model = StreamVGGT()
    ckpt_path = hf_hub_download(
        repo_id="lch01/StreamVGGT",
        filename="checkpoints.pth",
        revision="main",
    )
    ckpt = torch.load(ckpt_path, map_location="cpu")
    model.load_state_dict(ckpt, strict=True)
    model = model.to(device).eval()
    return model


def load_scenevggt():
    """Load SceneVGGT (uses VGGT architecture with same weights)."""
    _clean_sys_path()
    sys.path.insert(0, str(EXTERNAL / "SceneVGGT"))
    from vggt.models.vggt import VGGT
    model = VGGT.from_pretrained("facebook/VGGT-1B")
    model = model.to(device).eval()
    return model


def load_infinitevggt():
    """Load InfiniteVGGT (StreamVGGT with token budget)."""
    _clean_sys_path()
    sys.path.insert(0, str(EXTERNAL / "InfiniteVGGT" / "src"))
    from streamvggt.models.streamvggt import StreamVGGT
    from huggingface_hub import hf_hub_download
    model = StreamVGGT(total_budget=1_200_000)
    # Uses same StreamVGGT checkpoint
    ckpt_path = hf_hub_download(
        repo_id="lch01/StreamVGGT",
        filename="checkpoints.pth",
        revision="main",
    )
    ckpt = torch.load(ckpt_path, map_location="cpu")
    model.load_state_dict(ckpt, strict=True)
    model = model.to(device).eval()
    return model


def load_fastvggt():
    """Load FastVGGT (VGGT with token merging)."""
    _clean_sys_path()
    sys.path.insert(0, str(EXTERNAL / "FastVGGT"))
    from vggt.models.vggt import VGGT
    from huggingface_hub import hf_hub_download
    model = VGGT(merging=0, merge_ratio=0.9)
    ckpt_path = hf_hub_download(
        repo_id="facebook/VGGT_tracker_fixed",
        filename="model.pt",
    )
    ckpt = torch.load(ckpt_path, map_location="cpu")
    model.load_state_dict(ckpt, strict=False)
    model = model.to(device).eval()
    return model


LOADERS = {
    "VGGT": load_vggt_baseline,
    "StreamVGGT": load_streamvggt,
    "SceneVGGT": load_scenevggt,
    "InfiniteVGGT": load_infinitevggt,
    "FastVGGT": load_fastvggt,
}
print(f"Defined loaders for {len(LOADERS)} variants: {list(LOADERS.keys())}")

---
## 4. Benchmark Function <a id='4'></a>

In [ ]:
def benchmark_variant(
    name: str,
    model: torch.nn.Module,
    inference_fn,
    images: torch.Tensor,
    n_warmup: int = 2,
    n_runs: int = 5,
) -> dict:
    """Benchmark a single variant at a single sequence length.

    Args:
        name: Variant name for logging.
        model: The loaded model.
        inference_fn: Callable(model, images) -> output_dict.
        images: [N, 3, H, W] tensor on CPU, float32, [0,1].
        n_warmup: Warmup runs (not timed).
        n_runs: Timed runs.

    Returns:
        dict with latency_ms, peak_memory_mb, output_shapes, status.
    """
    N = images.shape[0]
    images_dev = images.to(device)

    try:
        # Warmup
        for _ in range(n_warmup):
            with torch.no_grad():
                with torch.amp.autocast(device.type, dtype=dtype):
                    _ = inference_fn(model, images_dev)
            if device.type == "cuda":
                torch.cuda.synchronize()

        # Timed runs
        latencies = []
        peak_mems = []
        output_shapes = None

        for _ in range(n_runs):
            reset_memory_stats()
            if device.type == "cuda":
                torch.cuda.synchronize()

            t0 = time.perf_counter()
            with torch.no_grad():
                with torch.amp.autocast(device.type, dtype=dtype):
                    output = inference_fn(model, images_dev)
            if device.type == "cuda":
                torch.cuda.synchronize()
            t1 = time.perf_counter()

            latencies.append((t1 - t0) * 1000)  # ms
            peak_mems.append(get_peak_memory_mb())

            # Capture output shapes on first run
            if output_shapes is None:
                output_shapes = {}
                for k, v in output.items():
                    if isinstance(v, torch.Tensor):
                        output_shapes[k] = tuple(v.shape)
                    elif isinstance(v, list) and len(v) > 0 and isinstance(v[0], torch.Tensor):
                        output_shapes[k] = f"list[{len(v)}] of {tuple(v[0].shape)}"

        return {
            "variant": name,
            "seq_len": N,
            "latency_ms": np.median(latencies),
            "latency_std_ms": np.std(latencies),
            "peak_memory_mb": max(peak_mems),
            "output_shapes": output_shapes,
            "status": "ok",
        }

    except torch.cuda.OutOfMemoryError:
        if device.type == "cuda":
            torch.cuda.empty_cache()
        return {
            "variant": name,
            "seq_len": N,
            "latency_ms": float("nan"),
            "latency_std_ms": float("nan"),
            "peak_memory_mb": float("nan"),
            "output_shapes": None,
            "status": "OOM",
        }
    except Exception as e:
        return {
            "variant": name,
            "seq_len": N,
            "latency_ms": float("nan"),
            "latency_std_ms": float("nan"),
            "peak_memory_mb": float("nan"),
            "output_shapes": None,
            "status": f"error: {e}",
        }

print("benchmark_variant() defined")

---
## 5. Per-Variant Inference Wrappers <a id='5'></a>

Each variant has a different forward API. These wrappers normalize them to
`fn(model, images) -> dict` where `images` is `[N, 3, H, W]` on device.

In [ ]:
def infer_vggt_baseline(model, images):
    """VGGT baseline: model(images) -> dict."""
    return model(images)


def infer_streamvggt(model, images):
    """StreamVGGT: model.inference(frames) where frames is list of dicts."""
    frames = [{"img": images[i].unsqueeze(0)} for i in range(images.shape[0])]
    output = model.inference(frames)
    # Normalize to dict with tensor values for shape inspection
    result = {}
    if hasattr(output, "ress") and output.ress:
        # Stack per-frame results into tensors where possible
        first = output.ress[0]
        for key in first:
            if isinstance(first[key], torch.Tensor):
                try:
                    result[key] = torch.stack([r[key] for r in output.ress])
                except (RuntimeError, KeyError):
                    result[key] = first[key]  # fallback: just first frame
    return result


def infer_scenevggt(model, images):
    """SceneVGGT: same forward API as VGGT baseline."""
    return model(images)


def infer_infinitevggt(model, images):
    """InfiniteVGGT: model.inference(frames) with token budget."""
    frames = [{"img": images[i].unsqueeze(0)} for i in range(images.shape[0])]
    output = model.inference(frames)
    result = {}
    if hasattr(output, "ress") and output.ress:
        first = output.ress[0]
        for key in first:
            if isinstance(first[key], torch.Tensor):
                try:
                    result[key] = torch.stack([r[key] for r in output.ress])
                except (RuntimeError, KeyError):
                    result[key] = first[key]
    return result


def infer_fastvggt(model, images):
    """FastVGGT: same forward API as VGGT baseline (with token merging)."""
    return model(images)


INFERENCE_FNS = {
    "VGGT": infer_vggt_baseline,
    "StreamVGGT": infer_streamvggt,
    "SceneVGGT": infer_scenevggt,
    "InfiniteVGGT": infer_infinitevggt,
    "FastVGGT": infer_fastvggt,
}
print(f"Defined inference wrappers for {len(INFERENCE_FNS)} variants")

---
## 6. Run Benchmarks <a id='6'></a>

For each variant: load model -> benchmark all sequence lengths -> unload.

In [ ]:
results = []

for variant_name in LOADERS:
    print(f"\n{'='*60}")
    print(f"Loading {variant_name}...")
    print(f"{'='*60}")

    try:
        model = LOADERS[variant_name]()
        print(f"  Model loaded on {device}")
    except Exception as e:
        print(f"  FAILED to load: {e}")
        for N in SEQ_LENGTHS:
            results.append({
                "variant": variant_name,
                "seq_len": N,
                "latency_ms": float("nan"),
                "latency_std_ms": float("nan"),
                "peak_memory_mb": float("nan"),
                "output_shapes": None,
                "status": f"load_error: {e}",
            })
        continue

    inference_fn = INFERENCE_FNS[variant_name]

    for N in SEQ_LENGTHS:
        print(f"  Benchmarking N={N}...", end=" ", flush=True)
        images_subset = all_images[:N]
        result = benchmark_variant(
            name=variant_name,
            model=model,
            inference_fn=inference_fn,
            images=images_subset,
            n_warmup=2,
            n_runs=5,
        )
        results.append(result)
        if result["status"] == "ok":
            print(f"latency={result['latency_ms']:.0f}ms, mem={result['peak_memory_mb']:.0f}MB")
        else:
            print(f"status={result['status']}")

    unload_model(model)
    print(f"  {variant_name} unloaded.")

print(f"\nTotal results: {len(results)}")

---
## 7. Results DataFrame <a id='7'></a>

In [ ]:
df = pd.DataFrame(results)
# Compute per-frame latency
df["latency_per_frame_ms"] = df["latency_ms"] / df["seq_len"]

display_cols = ["variant", "seq_len", "latency_ms", "latency_per_frame_ms",
                "peak_memory_mb", "status"]
display(df[display_cols].round(1))

---
## 8. Latency Plot <a id='8'></a>

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

df_ok = df[df["status"] == "ok"]

for variant in df_ok["variant"].unique():
    sub = df_ok[df_ok["variant"] == variant].sort_values("seq_len")
    ax1.plot(sub["seq_len"], sub["latency_ms"], marker="o", label=variant)
    ax2.plot(sub["seq_len"], sub["latency_per_frame_ms"], marker="o", label=variant)

ax1.set_xlabel("Sequence length (N frames)")
ax1.set_ylabel("Total latency (ms)")
ax1.set_title("Total Inference Latency vs Sequence Length")
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.set_xlabel("Sequence length (N frames)")
ax2.set_ylabel("Latency per frame (ms/frame)")
ax2.set_title("Per-Frame Latency vs Sequence Length")
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---
## 9. Memory Plot <a id='9'></a>

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

for variant in df_ok["variant"].unique():
    sub = df_ok[df_ok["variant"] == variant].sort_values("seq_len")
    ax.plot(sub["seq_len"], sub["peak_memory_mb"], marker="s", label=variant)

ax.set_xlabel("Sequence length (N frames)")
ax.set_ylabel("Peak GPU memory (MB)")
ax.set_title("Peak GPU Memory vs Sequence Length")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---
## 10. Combined Summary Bar Chart <a id='10'></a>

In [ ]:
key_ns = [10, 50, 100]
df_key = df_ok[df_ok["seq_len"].isin(key_ns)].copy()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

variants = df_key["variant"].unique()
n_variants = len(variants)
bar_width = 0.8 / n_variants
x_positions = np.arange(len(key_ns))

for i, variant in enumerate(variants):
    sub = df_key[df_key["variant"] == variant].set_index("seq_len")
    latencies = [sub.loc[n, "latency_ms"] if n in sub.index else 0 for n in key_ns]
    memories = [sub.loc[n, "peak_memory_mb"] if n in sub.index else 0 for n in key_ns]
    offset = (i - n_variants / 2 + 0.5) * bar_width
    axes[0].bar(x_positions + offset, latencies, bar_width, label=variant)
    axes[1].bar(x_positions + offset, memories, bar_width, label=variant)

for ax, ylabel, title in [
    (axes[0], "Latency (ms)", "Total Latency at Key Sequence Lengths"),
    (axes[1], "Peak Memory (MB)", "Peak GPU Memory at Key Sequence Lengths"),
]:
    ax.set_xticks(x_positions)
    ax.set_xticklabels([f"N={n}" for n in key_ns])
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.legend()
    ax.grid(True, alpha=0.3, axis="y")

plt.tight_layout()
plt.show()

---
## 11. Output Shape Consistency <a id='11'></a>

Compare output shapes across all variants at N=10 to check compatibility
for downstream world model integration.

In [ ]:
# Gather output shapes at N=10
df_n10 = df[df["seq_len"] == 10].copy()

print("Output shapes at N=10:\n")
for _, row in df_n10.iterrows():
    print(f"--- {row['variant']} (status: {row['status']}) ---")
    if row["output_shapes"]:
        for key, shape in row["output_shapes"].items():
            print(f"  {key:<25} {shape}")
    else:
        print("  (no output shapes available)")
    print()

# Build comparison table for common keys
shape_records = []
for _, row in df_n10.iterrows():
    if row["output_shapes"]:
        for key, shape in row["output_shapes"].items():
            shape_records.append({
                "variant": row["variant"],
                "output_key": key,
                "shape": str(shape),
            })

if shape_records:
    df_shapes = pd.DataFrame(shape_records)
    shape_pivot = df_shapes.pivot(index="output_key", columns="variant", values="shape")
    print("\nShape comparison pivot table:")
    display(shape_pivot)

    # Flag mismatches
    print("\nMismatch check:")
    for key in shape_pivot.index:
        unique_shapes = shape_pivot.loc[key].dropna().unique()
        if len(unique_shapes) > 1:
            print(f"  WARNING: '{key}' has different shapes: {unique_shapes}")
        else:
            print(f"  OK: '{key}' consistent across all variants")

---
## 12. Summary & Recommendation <a id='12'></a>

### Trade-off Analysis

| Criterion | Best candidate | Notes |
|-----------|---------------|-------|
| **Lowest latency** | FastVGGT | Token merging reduces computation |
| **Lowest memory** | StreamVGGT / InfiniteVGGT | Streaming processes frames incrementally |
| **Long sequences** | InfiniteVGGT | Token budget prevents OOM on N>100 |
| **Output compatibility** | VGGT / SceneVGGT / FastVGGT | Same output dict format |
| **Scene understanding** | SceneVGGT | Built-in SLAM modules, but adds complexity |

### Recommendation

**Fill in after running on GPU:** Compare the actual numbers from the benchmark above.
Key questions:
1. Which variants can handle N=100+ without OOM on H100 80GB?
2. What is the per-frame latency overhead of streaming vs batch?
3. Are output shapes compatible with DreamerV3 observation encoder?

The selected variant will be integrated as the 3D feature encoder in the
DreamerV3 world model pipeline for HM3D ObjectNav experiments.